In [1]:
import sys
from pathlib import Path

# Locate repo root by searching upward for requirements.txt
_cwd = Path.cwd()
REPO_ROOT = next(
    (p for p in [_cwd, *_cwd.parents] if (p / 'requirements.txt').exists()),
    _cwd,
)
sys.path.insert(0, str(REPO_ROOT.parent))  # adds parent so "from aneu_ghd import" works

import trimesh
from aneu_ghd import (
    load_landmarks, normalize_lm, compute_landmarks_from_rings,
    anatomy_align, cpd_align,
    FitConfig, FitResult, ghd_fit,
    compute_eigenvectors,
)

import numpy as np
import torch

from pytorch3d.io import load_obj
from pytorch3d.structures import Meshes
from pytorch3d.ops import sample_points_from_meshes
from pytorch3d.loss import chamfer_distance

# ─────────────────────────────────────────────
# STEP 1: Load meshes (PyTorch3D)
# ─────────────────────────────────────────────
canonical_path = str(REPO_ROOT / "canonical/canonical.obj")
# ── USER: set path to your target mesh ──────────────────────────────────────
target_path    = str(REPO_ROOT.parent / "GHD2/Checkpoints/Alignment/C0005/final_aligned.obj")

V_canon, F_canon, _ = load_obj(canonical_path)
V_tgt,   F_tgt,   _ = load_obj(target_path)

V = V_canon.cpu().numpy().astype(np.float64)
F = F_canon.verts_idx.cpu().numpy().astype(np.int64)

V_target = V_tgt.cpu().numpy().astype(np.float64)
F_target = F_tgt.verts_idx.cpu().numpy().astype(np.int64)

print("Canonical:", V.shape, F.shape)
print("Target:   ", V_target.shape, F_target.shape)

# ─────────────────────────────────────────────
# STEP 1b: Centroid + max-norm normalization
#
# Subtract centroid → both shapes centred at origin.
# Divide by max vertex distance → all vertices inside unit sphere (max = 1).
# CPD starts from identity (both shapes overlapping) → reliable convergence.
# ─────────────────────────────────────────────
c_can   = V.mean(axis=0)
c_tgt   = V_target.mean(axis=0)

V_n     = V        - c_can
V_tgt_n = V_target - c_tgt

s_can   = np.max(np.linalg.norm(V_n,     axis=1))   # max-norm → unit sphere
s_tgt   = np.max(np.linalg.norm(V_tgt_n, axis=1))
V_n    /= s_can
V_tgt_n /= s_tgt

print(f"Canonical: centroid={c_can.round(3)}, scale={s_can:.3f} mm, "
      f"max_r={np.linalg.norm(V_n, axis=1).max():.3f}")
print(f"Target:    centroid={c_tgt.round(3)}, scale={s_tgt:.3f} mm, "
      f"max_r={np.linalg.norm(V_tgt_n, axis=1).max():.3f}")

# ─────────────────────────────────────────────
# STEP 2-3: Graph Fourier basis
# Mixed Laplacian (cotangent + inv-length + uniform) eigenvectors.
# See aneu_ghd/utils.py for implementation details.
# ─────────────────────────────────────────────
U = compute_eigenvectors(V_n.astype(np.float64), F, n_basis=121)
print(f"U shape: {U.shape}")

In [2]:
# ── Save canonical eigenvectors (run once; reused by scripts/fit_case.py) ────
# After running this cell, pass the saved path to --eigenvecs in fit_case.py
# to skip the ~1 min Laplacian build on every run.
import os
_cache_dir = os.path.dirname(canonical_path)
np.save(os.path.join(_cache_dir, 'eigenvectors.npy'), U)
np.save(os.path.join(_cache_dir, 'centroid.npy'),    c_can)
np.save(os.path.join(_cache_dir, 'scale.npy'),       np.array(s_can))
print(f'Saved U {U.shape} → {_cache_dir}/eigenvectors.npy')
print(f'Saved centroid and scale for reference.')


In [3]:
# ── STEP 4: Device setup ───────────────────────────────────────────────────────────────────
device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


Using device: cuda:2


In [4]:
# ── STEP 4a: Load landmarks & compute vertex indices ─────────────────────────────────────
# Note: meshes from final_aligned.obj are already orientation-aligned by the cutting pipeline.
# anatomy_align() is commented out — uncomment if your mesh is NOT pre-aligned.
LANDMARKS_DIR = str(REPO_ROOT)
CANONICAL_ID  = 'canonical'
TARGET_ID     = 'GHD2/Checkpoints/Alignment/C0005'

lm_can   = load_landmarks(LANDMARKS_DIR, CANONICAL_ID)
lm_tgt   = load_landmarks(LANDMARKS_DIR, TARGET_ID)
lm_can_s = normalize_lm(lm_can, c_can, s_can)
lm_tgt_s = normalize_lm(lm_tgt, c_tgt, s_tgt)

# Landmark vertex indices for GHD landmark loss
idx_lm_dome = int(np.argmin(np.linalg.norm(V_n - lm_can_s['dome'], axis=1)))
k_cap       = 20
cap_up_idxs = np.argsort(np.linalg.norm(V_n - lm_can_s['cap_up'],   axis=1))[:k_cap]
cap_dn_idxs = np.argsort(np.linalg.norm(V_n - lm_can_s['cap_down'], axis=1))[:k_cap]

# ── Anatomy alignment (use for non-pre-aligned meshes) ──────────────────────
# Shared chamfer helper (pytorch3d, 5000 sampled pts — same metric as ghd_fit)
def _chamfer_p3d(Va, Fa, Vb, Fb, n=5000):
    ma = Meshes(verts=[torch.tensor(Va, dtype=torch.float32, device=device)],
                faces=[torch.tensor(Fa, dtype=torch.int64,   device=device)])
    mb = Meshes(verts=[torch.tensor(Vb, dtype=torch.float32, device=device)],
                faces=[torch.tensor(Fb, dtype=torch.int64,   device=device)])
    with torch.no_grad():
        xa = sample_points_from_meshes(ma, n)
        xb = sample_points_from_meshes(mb, n)
        return chamfer_distance(xa, xb)[0].item()
print(f'  Chamfer before anatomy: {_chamfer_p3d(V_n, F, V_tgt_n, F_target):.6f}')
V_n = anatomy_align(V_n, lm_can_s, lm_tgt_s)
print(f'  Chamfer after  anatomy: {_chamfer_p3d(V_n, F, V_tgt_n, F_target):.6f}')

print(f'  Canonical: r_up={float(lm_can["r_up_mm"]):.3f} mm  '
      f'r_dn={float(lm_can["r_down_mm"]):.3f} mm  z_flipped={bool(lm_can["z_was_flipped"])}')
print(f'  Target:    r_up={float(lm_tgt["r_up_mm"]):.3f} mm  '
      f'r_dn={float(lm_tgt["r_down_mm"]):.3f} mm  z_flipped={bool(lm_tgt["z_was_flipped"])}')
print(f'  Landmark indices: dome={idx_lm_dome},  cap k={k_cap}')


  Chamfer before anatomy: 0.119537
  Chamfer after  anatomy: 0.018037
  Canonical: r_up=1.365 mm  r_dn=1.417 mm  z_flipped=False
  Target:    r_up=0.000 mm  r_dn=4.069 mm  z_flipped=False
  Landmark indices: dome=2320,  cap k=20


In [5]:
# ── STEP 4b: CPD rigid refinement ────────────────────────────────────────────────────
print(f'Before CPD: {_chamfer_p3d(V_n, F, V_tgt_n, F_target):.6f}')

V_aligned_np, R_cpd, t_cpd = cpd_align(V_n, V_tgt_n)

print(f'CPD: det(R)={np.linalg.det(R_cpd):.6f},  |t|={np.linalg.norm(t_cpd):.6f}')
print(f'After  CPD: {_chamfer_p3d(V_aligned_np.astype(np.float32), F, V_tgt_n, F_target):.6f}')


Before CPD: 0.017948


CPD: det(R)=1.000000,  |t|=0.068342
After  CPD: 0.014725


In [6]:
# ── STEP 4d: Set GHD initialiser ────────────────────────────────────────────────────────
V_init_np = V_aligned_np.astype(np.float32)
print('V_init shape:', V_init_np.shape, '  dtype:', V_init_np.dtype)


V_init shape: (2610, 3)   dtype: float32


In [7]:
# ── STEP 5: Configure GHD fitting ───────────────────────────────────────────────────────
# Override any weight with: cfg = FitConfig(lambda_landmark=0.0)
cfg = FitConfig(
    n_iter             = 12_000,
    lambda_chamfer     = 1.0,
    lambda_chamfer_n1  = 0.5,
    lambda_occupancy   = 1.0,
    lambda_laplacian   = 1e-2,
    lambda_consistency = 0.3,
    lambda_rigid_start = 3.0,
    lambda_rigid_end   = 0.01,
    rigid_decay_frac   = 0.7,
    lambda_landmark    = 0.1,
    lm_taper_frac      = 0.3,
    device             = 'cuda:2',
)
print(cfg)


FitConfig(n_iter=12000, lr=0.001, eta_min=1e-06, fit_R=True, fit_s=True, fit_T=True, num_samples=5000, n_dvs=10000, n_dvs_oversamp=60000, num_dvs_sample=12000, NP_ratio=1.0, lambda_chamfer=1.0, lambda_chamfer_n1=0.5, lambda_occupancy=1.0, lambda_laplacian=0.01, lambda_consistency=0.3, lambda_rigid_start=3.0, lambda_rigid_end=0.01, rigid_decay_frac=0.7, lambda_edge=0.1, lambda_landmark=0.1, lm_taper_frac=0.3, log_every=200, save_every=1000, out_dir='ghd_debug_outputs', converge_threshold=0.95, device='cuda:2')


In [8]:
# ── STEP 6: Run GHD fitting ─────────────────────────────────────────────────────────────
V0_np = V_init_np.copy()   # kept for post-fit displacement visualisation

result = ghd_fit(
    V_init      = V_init_np,
    F           = F,
    V_tgt       = V_tgt_n.astype(np.float32),
    F_tgt       = F_target,
    U           = U.astype(np.float32),
    cfg         = cfg,
    lm_can_s    = lm_can_s,
    lm_tgt_s    = lm_tgt_s,
    idx_lm_dome = idx_lm_dome,
    cap_up_idxs = cap_up_idxs,
    cap_dn_idxs = cap_dn_idxs,
)

V_final = result.verts   # (N, 3) fitted mesh in normalised space


Preparing DVS samples...
  trimesh.contains: 0.1s
  DVS: 10361 interior pts, 49639 exterior pts


/home/yaplab2/miniconda3/envs/new/lib/python3.11/site-packages/pytorch3d/ops/laplacian_matrices.py:130: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /opt/conda/conda-bld/pytorch_1720538437738/work/torch/csrc/utils/tensor_new.cpp:641.)
  L = torch.sparse.FloatTensor(idx, cot.view(-1), (V, V))


Baseline Chamfer: 0.014335
Starting GHD fitting loop...
Iter     0 | Chamfer: 0.014450 | N1: 0.1870 | Occ: 0.0851 | Lap: 2.32e-03 | Cons: 5.28e-03 | Edge: 0.0000 | Rigid: 0.0000 (w=3.000) | LM: 0.0228 | Total: 0.196966 | lr: 1.00e-03 | 6.6 it/s


Iter   200 | Chamfer: 0.013901 | N1: 0.1707 | Occ: 0.0806 | Lap: 2.30e-03 | Cons: 5.28e-03 | Edge: 0.0000 | Rigid: 0.0002 (w=2.929) | LM: 0.0220 | Total: 0.184206 | lr: 9.99e-04 | 22.3 it/s
Iter   400 | Chamfer: 0.013584 | N1: 0.1711 | Occ: 0.0806 | Lap: 2.30e-03 | Cons: 5.28e-03 | Edge: 0.0000 | Rigid: 0.0002 (w=2.858) | LM: 0.0207 | Total: 0.183928 | lr: 9.97e-04 | 22.4 it/s
Iter   600 | Chamfer: 0.013609 | N1: 0.1701 | Occ: 0.0806 | Lap: 2.30e-03 | Cons: 5.28e-03 | Edge: 0.0000 | Rigid: 0.0002 (w=2.786) | LM: 0.0194 | Total: 0.183291 | lr: 9.94e-04 | 22.5 it/s
Iter   800 | Chamfer: 0.013478 | N1: 0.1714 | Occ: 0.0806 | Lap: 2.30e-03 | Cons: 5.28e-03 | Edge: 0.0000 | Rigid: 0.0002 (w=2.715) | LM: 0.0181 | Total: 0.183695 | lr: 9.89e-04 | 22.5 it/s
Iter  1000 | Chamfer: 0.013859 | N1: 0.1709 | Occ: 0.0806 | Lap: 2.30e-03 | Cons: 5.28e-03 | Edge: 0.0000 | Rigid: 0.0002 (w=2.644) | LM: 0.0168 | Total: 0.183699 | lr: 9.83e-04 | 22.5 it/s
Iter  1200 | Chamfer: 0.013941 | N1: 0.1713 | Occ:

In [9]:
# ── STEP 7b: Fit summary ───────────────────────────────────────────────────────────────────
print(f'Chamfer (best)  : {result.chamfer_best:.6f}  @ iter {result.best_iter}')
print(f'Chamfer (final) : {result.chamfer_final:.6f}')
print(f'Dice score      : {result.dice:.4f}')
print(f'GAR             : {result.gar:.4f}')
print(f'Converged       : {result.converged}')

m = trimesh.Trimesh(vertices=V_final, faces=F, process=False)
print('watertight          :', m.is_watertight)
print('winding consistent  :', m.is_winding_consistent)
print('Euler number        :', m.euler_number)


Chamfer (best)  : 0.000509  @ iter 10112
Chamfer (final) : 0.000533
Dice score      : 0.9987
GAR             : 0.9624
Converged       : True
watertight          : True
winding consistent  : True
Euler number        : 2


In [10]:
# ── STEP 8: Export fitted mesh ──────────────────────────────────────────────────────────
from pathlib import Path

result_mesh = trimesh.Trimesh(vertices=V_final, faces=F, process=False)

out_path = REPO_ROOT / 'fitting_results/C0005/ghd_fitted.obj'
out_path.parent.mkdir(parents=True, exist_ok=True)
result_mesh.export(str(out_path))
print(f'Saved: {out_path}')


In [18]:
import trimesh

mesh_fit = trimesh.Trimesh(vertices=V_final, faces=F, process=False)
mesh_tgt = trimesh.Trimesh(vertices=V_tgt_n, faces=F_target, process=False)

mesh_fit.visual.face_colors = [255, 0, 0, 120]   # red  = fitted
mesh_tgt.visual.face_colors = [0, 0, 255, 120]   # blue = target

trimesh.Scene([mesh_fit, mesh_tgt]).show()


In [20]:
import numpy as np
import trimesh
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# assume V0, V_final, F, and optionally V_tgt_n, F_target already exist
Vf_np = np.asarray(V_final)
F_np  = np.asarray(F)

# displacement stats
disp = np.linalg.norm(Vf_np - V0_np, axis=1)
print(f"max displacement: {disp.max():.6f}")
print(f"mean displacement: {disp.mean():.6f}")
print(f"std displacement: {disp.std():.6f}")

# blue -> red colors per vertex
norm = mcolors.Normalize(vmin=disp.min(), vmax=disp.max())
rgba = (cm.get_cmap('coolwarm')(norm(disp)) * 255).astype(np.uint8)

mesh_fit = trimesh.Trimesh(vertices=Vf_np, faces=F_np, process=False)
mesh_fit.visual.vertex_colors = rgba

# optional: overlay target in transparent blue
scene_meshes = [mesh_fit]
if 'V_tgt_n' in globals() and 'F_target' in globals():
    mesh_tgt = trimesh.Trimesh(vertices=V_tgt_n, faces=F_target, process=False)
    mesh_tgt.visual.face_colors = [0, 0, 255, 80]
    scene_meshes.append(mesh_tgt)

scene = trimesh.Scene(scene_meshes)
scene.show()   # interactive rotate/zoom/pan


max displacement: 0.351842
mean displacement: 0.160704
std displacement: 0.075560


/tmp/ipykernel_1458107/1661983847.py:18: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  rgba = (cm.get_cmap('coolwarm')(norm(disp)) * 255).astype(np.uint8)


In [11]:
# ── Test compute_landmarks_from_rings() against loaded landmark file ──────────
# Run this cell to validate that the landmark-free path gives consistent
# results before using it on meshes without a .npz file.
#
# Inputs the future cutting pipeline will provide:
#   - surface mesh (V, F)
#   - inlet_ring_nodes  : boundary ring nodes of inlet  opening
#   - outlet_ring_nodes : boundary ring nodes of outlet opening
#   - dome_pts          : segmented dome point cloud

from collections import defaultdict

# ── Step 1: Extract boundary ring nodes from mesh open caps ───────────────────
# Count how many faces each edge belongs to; boundary edges belong to only 1.
edge_count = defaultdict(int)
for face in F:
    for i in range(3):
        e = tuple(sorted([int(face[i]), int(face[(i+1)%3])]))
        edge_count[e] += 1
boundary_edges = [e for e, cnt in edge_count.items() if cnt == 1]

# Split into connected rings
adj = defaultdict(set)
for e in boundary_edges:
    adj[e[0]].add(e[1])
    adj[e[1]].add(e[0])

visited, rings = set(), []
for start in set(v for e in boundary_edges for v in e):
    if start in visited:
        continue
    ring, stack = [], [start]
    while stack:
        v = stack.pop()
        if v in visited: continue
        visited.add(v); ring.append(v)
        stack.extend(adj[v] - visited)
    rings.append(np.array(ring))

print(f'Boundary rings: {len(rings)} found')
for i, r in enumerate(rings):
    print(f'  Ring {i}: {len(r)} vertices,  centroid={V[r].mean(0).round(4)}')

# Assign inlet / outlet (the function handles which is larger automatically)
inlet_ring_nodes  = V[rings[0]]   # (N, 3) world coords
outlet_ring_nodes = V[rings[1]]   # (M, 3) world coords

# ── Step 2: Dome point cloud ───────────────────────────────────────────────────
# Use vertices far from both cap centroids as proxy for the segmented dome.
# In production: pass the actual segmented dome surface from the cutting pipeline.
P_up = inlet_ring_nodes.mean(axis=0)
P_dn = outlet_ring_nodes.mean(axis=0)
cap_centers = np.stack([P_up, P_dn])                 # (2, 3)
dist_to_caps = np.linalg.norm(
    V[:, None, :] - cap_centers[None, :, :], axis=2
).min(axis=1)                                        # (N,) min dist to either cap
dome_threshold = np.percentile(dist_to_caps, 40)     # top 60% furthest from caps
dome_pts = V[dist_to_caps > dome_threshold]
print(f'Dome proxy pts: {len(dome_pts)}')

# ── Step 3: Compute landmarks from rings ──────────────────────────────────────
lm_computed = compute_landmarks_from_rings(
    inlet_ring_nodes  = inlet_ring_nodes,
    outlet_ring_nodes = outlet_ring_nodes,
    dome_pts          = dome_pts,
)

# ── Step 4: Normalize both and compare ────────────────────────────────────────
# Use canonical mesh centroid/scale (same mesh, so directly comparable)
lm_computed_s = normalize_lm(lm_computed, c_can, s_can)

print()
print('─' * 55)
print(f'{"Landmark":<12}  {"Loaded (file)":>20}  {"Computed (rings)":>20}')
print('─' * 55)
for key in ['neck', 'dome', 'cap_up', 'cap_down']:
    a = lm_can_s[key].round(4)
    b = lm_computed_s[key].round(4)
    diff = np.linalg.norm(a - b)
    print(f'{key:<12}  {str(a):>20}  {str(b):>20}  |diff|={diff:.4f}')
print('─' * 55)
print(f'z_was_flipped  loaded={bool(lm_can["z_was_flipped"])}   '
      f'computed={bool(lm_computed["z_was_flipped"])}')
print()
print('Note: small differences are expected (dome proxy vs exact dome).')
print('If z_was_flipped matches and neck/dome directions are consistent → OK.')


Boundary rings: 0 found


IndexError: list index out of range